In [0]:
CREATE VOLUME IF NOT EXISTS workspace.default.streaming_volume;

In [0]:
%python
from pyspark.sql import functions as F

input_path = "/Volumes/workspace/default/streaming_volume/input"
schema_path = "/Volumes/workspace/default/streaming_volume/schema"
checkpoint_path = "/Volumes/workspace/default/streaming_volume/checkpoint_v2"

df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .load(input_path)
)

# Replace spaces with underscores
for col in df.columns:
    df = df.withColumnRenamed(col, col.replace(" ", "_"))

df = df.withColumn("load_date", F.current_date())

query = (
    df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable("workspace.default.streaming_table")
)

query.awaitTermination()